# Natural forgetting: overnight geometry and gradient study

Ordinary A→B training, without activation interventions. The worker runs separately; refresh status manually. Keep every supplied `.py` file next to this notebook. See README for the exact measurements and limitations.

In [ ]:
from pathlib import Path
import natural_geometry as study

SOURCE = study.discover_source() or "/home/ubuntu/1/runs/olmo_association_v1"
OUTPUT = Path.cwd() / "runs" / "olmo_natural_geometry_v1"
print("Source:", SOURCE)

## Settings
The default is an 11.5-hour session. It rotates between seeds every eight updates. Leave the existing original training settings unchanged for comparable trajectories.

In [ ]:
SETTINGS = study.defaults(SOURCE, OUTPUT)
SETTINGS.update(hours=11.5, device="cuda:0", threads=8)
# Optional BEFORE first launch:
# SETTINGS["curvature_every"] = 0  # skips expensive mixed Hessian; keeps other diagnostics
# SETTINGS["geometry_every"] = 8  # less frequent activation/topology snapshots

study.validate(SETTINGS)
print("Seeds:", SETTINGS["seeds"], "B updates:", SETTINGS["start_step"]+1, "to", SETTINGS["end_step"])
print("Output:", SETTINGS["output"])

## Start or resume
Run this cell once. Re-running while active is harmless. Scientific settings changes create a fresh sibling output directory automatically; the old results remain available.

In [ ]:
RUN_OUTPUT = study.launch(SETTINGS)

## Manual status refresh
Re-run this cell whenever you want an update. Last progress refers to worker activity, not the time you refreshed this cell. A long derivative/checkpoint operation can keep the same phase for a while.

In [ ]:
_ = study.refresh(SETTINGS["output"])

## Print numbers and display the plot
This reads saved data; it does not launch another experiment. Early in the run there may be no measured updates yet.

In [ ]:
study.show(SETTINGS["output"])

## Stop / fresh restart controls
These functions are defined without stopping anything when you Run All. To stop, call `stop_run()` in a new cell. Wait until status says `alive: false`. Resume using the Start cell. Use `fresh_run()` only when you want a new output directory.

In [ ]:
def stop_run():
    study.stop(SETTINGS["output"])

def fresh_run():
    return study.restart(SETTINGS)

## Export when ready
The worker exports automatically at session end. To export current saved measurements earlier, call `export_results()` in a new cell. Checkpoints and raw NPZ arrays are excluded from the share ZIP.

In [ ]:
def export_results():
    path = study.export(SETTINGS["output"])
    print("Share ZIP:", path)
    return path